# ∇ 与 Δ 的区别

| 符号    | 名称          | 数学含义                      | 直观理解    | RL 中的例子                                                                              |
| :---- | :---------- | :------------------------ | :------ | :----------------------------------------------------------------------------------- |
| **∇** | Nabla / Del | **梯度算子**，表示**变化率**或**导数** | "坡度有多陡" | $\nabla_\theta \log \pi_\theta(a)$ —— 策略参数 $\theta$ 微小变化时，对数概率的变化率                   |
| **Δ** | Delta       | **差分**，表示**两个状态的差值**      | "变了多少"  | $\Delta \log p = \log p_{\text{RL}} - \log p_{\text{Base}}$ —— RL 模型与 Base 模型的对数概率之差 |

## 一句话区分
- ∇ 回答"怎么变"（<font color='red'>方向 + 速率</font>），
- Δ 回答"<font color='blue'>变了多少</font>"（起点到终点的差距）。

# RL 训练中用到的矩阵/张量操作方法
以下是 RLHF/PPO/GRPO 等算法中高频出现的 PyTorch 矩阵操作：
##  概率计算与归一化

In [ ]:
# Softmax / Log-Softmax：将 logits 转为概率
log_probs = F.log_softmax(logits, dim=-1)      # [B, T, V]
probs = F.softmax(logits, dim=-1)              # [B, T, V]

# 提取特定 token 的对数概率
token_logp = torch.gather(log_probs, dim=2, 
                            index=token_ids.unsqueeze(-1)).squeeze(-1)  # [B, T]

## 掩码操作（处理变长序列）

In [ ]:
# Padding 掩码：无效位置置零
masked = tensor * attention_mask                 # 逐元素乘法
masked_sum = (tensor * mask).sum(dim=1)        # 屏蔽后求和
masked_mean = (tensor * mask).sum(dim=1) / mask.sum(dim=1)  # 屏蔽后平均

# 条件选择
advantages = torch.where(mask, advantages, torch.zeros_like(advantages))

## 损失聚合（序列级 / Prompt 级）

In [ ]:
# Token 级 → 序列级（Masked Sum）
seq_log_ratio = (log_diffs * mask).sum(dim=1)    # [B]

# 序列级 → Prompt 级（GRPO 中多个 response 对应一个 prompt）
prompt_loss = losses.view(num_prompts, num_per_prompt).mean(dim=1)

## 矩阵乘法与批量运算

In [ ]:
# 注意力机制中的 QK^T
scores = torch.matmul(Q, K.transpose(-2, -1)) / sqrt(d_k)   # [B, H, T, T]

# 线性层（策略网络、价值网络）
logits = torch.matmul(hidden_states, weight.t()) + bias        # [B, T, V]

# 批量矩阵乘法（BMM）
output = torch.bmm(A, B)                                      # [B, M, K] @ [B, K, N]

## 裁剪与约束

In [ ]:
# PPO Ratio Clip：限制策略更新幅度
clipped_ratio = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)

# 梯度裁剪：防止梯度爆炸
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# 值裁剪（Value Clipping）
value_clipped = old_values + torch.clamp(new_values - old_values, -clip_range, clip_range)

## 范数与归一化

In [ ]:
# L2 范数（梯度裁剪、优势归一化）
norm = torch.norm(grads, p=2)                  # 梯度总长度
advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)  # Batch 归一化

## 拼接与扩展

In [ ]:
# 拼接 prompt + response（用于前向传播）
full_ids = torch.cat([prompt_ids, response_ids], dim=1)       # [B, prompt_len + gen_len]

# 广播：将标量/向量扩展到矩阵维度
log_probs = log_probs + temperature_bias                      # [B, T, V] + [1, 1, V]

## 比较与选择

In [ ]:
# PPO Loss：取 clipped 和 unclipped 的最小值
loss = -torch.min(ratio * advantages, clipped_ratio * advantages).mean()

# 生成时的采样
next_token = torch.multinomial(probs, num_samples=1)           # 按概率采样
next_token = torch.argmax(logits, dim=-1)                     # 贪心解码

## 对角线与迹

In [ ]:
# GAE（广义优势估计）中的折扣累积
returns = torch.zeros_like(rewards)
for t in reversed(range(T)):
    returns[t] = rewards[t] + gamma * returns[t+1] * (1 - dones[t])

## Einsum（爱因斯坦求和约定）

In [ ]:
# 批量注意力计算
scores = torch.einsum('bhid,bhjd->bhij', Q, K)   # [B, H, T, D] @ [B, H, T, D] → [B, H, T, T]

# 一张图总结 RL 中的矩阵操作流

In [ ]:
输入: prompt_ids [B, L_p] + response_ids [B, L_r]
           ↓
    ┌─────────────────┐
    │  cat → [B, L_p+L_r] │
    └────────┬────────┘
             ↓
    ┌─────────────────┐
    │  model.forward() │  → logits [B, L, V]
    └────────┬────────┘
             ↓
    ┌─────────────────┐
    │ log_softmax(dim=-1)│ → log_probs [B, L, V]
    └────────┬────────┘
             ↓
    ┌─────────────────┐
    │ gather(实际token) │ → token_logp [B, L]
    └────────┬────────┘
             ↓
    ┌─────────────────┐
    │  Δlogp = logp_RL - logp_old │
    └────────┬────────┘
             ↓
    ┌─────────────────┐
    │  mask * sum(dim=1) │ → seq_log_ratio [B]
    └────────┬────────┘
             ↓
    ┌─────────────────┐
    │  exp → ratio, clamp → clip  │
    └────────┬────────┘
             ↓
    ┌─────────────────┐
    │  loss = min(ratio*A, clip*A) │
    └────────┬────────┘
             ↓
    ┌─────────────────┐
    │  backward() + clip_grad_norm │
    └────────┬────────┘
             ↓
         optimizer.step()